# 07 — Train the FNO

This notebook is self-contained after data acquisition. It rebuilds the chronological splits and loaders, trains the FNO, monitors validation loss, and saves both the best checkpoint and its experiment metadata.

In [ ]:
from pathlib import Path
import json
import platform
import random

import numpy as np
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from tqdm.auto import trange

from oisst_fno.data import ForecastSpec, SSTWindowDataset, Standardizer, open_oisst, temporal_split
from oisst_fno.metrics import masked_mse_loss, parameter_count
from oisst_fno.model import FNO2d

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = sorted((ROOT / "data" / "raw").glob("oisst_*_ne_atlantic.nc"))[-1]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("device:", DEVICE)

In [ ]:
TRAIN_END = "2024-12-31"
VALIDATION_END = "2025-12-31"
SPEC = ForecastSpec(lookback_days=14, horizon_days=7)
BATCH_SIZE = 16

sst = open_oisst(DATA_PATH)["sst"]
train_da, val_da, _ = temporal_split(sst, TRAIN_END, VALIDATION_END)
scaler = Standardizer.fit(train_da.values)
train_ds = SSTWindowDataset(scaler.transform(train_da.values), SPEC)
val_ds = SSTWindowDataset(scaler.transform(val_da.values), SPEC)


def collate_with_mask(batch):
    xs, ys, masks = zip(*batch)
    x = torch.stack(xs)
    y = torch.stack(ys)
    mask = torch.stack(masks)
    return torch.cat((x, mask), dim=1), y, mask

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_with_mask,
    num_workers=0,
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_with_mask,
    num_workers=0,
)
print(len(train_ds), len(val_ds))

In [ ]:
MODEL_CONFIG = {
    "in_channels": SPEC.lookback_days + 1,
    "out_channels": 1,
    "width": 48,
    "modes_y": 16,
    "modes_x": 16,
    "depth": 4,
    "padding": 8,
}

model = FNO2d(**MODEL_CONFIG).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
EPOCHS = 50
PATIENCE = 8
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
print(f"trainable parameters: {parameter_count(model):,}")

In [ ]:
def run_epoch(model, loader, *, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    total_examples = 0

    for x, y, mask in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        mask = mask.to(DEVICE)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            prediction = model(x)
            loss = masked_mse_loss(prediction, y, mask)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite loss detected: {loss.item()}")
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

        batch_size = x.shape[0]
        total_loss += float(loss.detach()) * batch_size
        total_examples += batch_size

    return total_loss / max(total_examples, 1)

In [ ]:
BEST_PATH = ROOT / "artifacts" / "models" / "fno_best.pt"
HISTORY_PATH = ROOT / "artifacts" / "metrics" / "fno_training_history.json"
CONFIG_PATH = ROOT / "artifacts" / "metrics" / "fno_experiment.json"
BEST_PATH.parent.mkdir(parents=True, exist_ok=True)
HISTORY_PATH.parent.mkdir(parents=True, exist_ok=True)

best_val = float("inf")
stale_epochs = 0
history = []

for epoch in trange(EPOCHS):
    train_loss = run_epoch(model, train_loader, optimizer=optimizer)
    val_loss = run_epoch(model, val_loader)
    current_lr = float(optimizer.param_groups[0]["lr"])
    scheduler.step()

    history.append(
        {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "learning_rate": current_lr,
        }
    )

    if val_loss < best_val:
        best_val = val_loss
        stale_epochs = 0
        torch.save(model.state_dict(), BEST_PATH)
    else:
        stale_epochs += 1
        if stale_epochs >= PATIENCE:
            break

HISTORY_PATH.write_text(json.dumps(history, indent=2), encoding="utf-8")
experiment = {
    "data_path": str(DATA_PATH),
    "train_end": TRAIN_END,
    "validation_end": VALIDATION_END,
    "lookback_days": SPEC.lookback_days,
    "horizon_days": SPEC.horizon_days,
    "batch_size": BATCH_SIZE,
    "seed": SEED,
    "scaler": {"mean": scaler.mean, "std": scaler.std},
    "model": MODEL_CONFIG,
    "optimizer": {"name": "AdamW", "lr": 2e-3, "weight_decay": 1e-4},
    "epochs_requested": EPOCHS,
    "patience": PATIENCE,
    "best_validation_mse_standardized": best_val,
    "python": platform.python_version(),
    "torch": torch.__version__,
}
CONFIG_PATH.write_text(json.dumps(experiment, indent=2), encoding="utf-8")
print("best validation MSE:", best_val)
print(BEST_PATH)

In [ ]:
import matplotlib.pyplot as plt

train_curve = [row["train_loss"] for row in history]
val_curve = [row["val_loss"] for row in history]
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_curve, label="train")
ax.plot(val_curve, label="validation")
ax.set_xlabel("epoch")
ax.set_ylabel("masked standardized MSE")
ax.legend()
ax.set_title("FNO training history")
plt.show()